# Lecture16: Delta-Neutral RL ?? KuCoin (Colab)

??????? ??????? ?????????? ?????? ????????:
1. ????????? OHLCV Spot/Futures ? KuCoin.
2. ?????? ?????? ?? basis z-score.
3. ?????????? rule-based ????????? ? RL-?????? (PPO).
4. ???????????? `latest_forecast_signal_kucoin_rl.json` ??? ????????????? ????.

?????? ?????????:
- ????: `z > entry_z` -> `BUY spot + SELL futures`.
- ?????: `|z| < exit_z` -> ????????? ??? ????.
- ????: `net delta ? 0`.


In [ ]:
# ????????? ?????????
!pip -q install numpy pandas matplotlib requests gymnasium stable-baselines3


In [ ]:
import json
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ???????????
SPOT_SYMBOL = "NEAR-USDT"
FUTURES_SYMBOL = "NEARUSDTM"
TF_MIN = 1
LIMIT = 1200

# ?????? delta-neutral
BASIS_WINDOW = 60
ENTRY_Z = 1.5
EXIT_Z = 0.3

# ?????? ??????? (??? ? ????? ????)
CONTRACT_SIZE = 0.1          # 1 futures ???????? = 0.1 NEAR
MAX_SPOT_NOTIONAL_USDT = 1.0 # ???????????? ?????? ?????
SPOT_MIN_FUNDS = 0.1
SPOT_MIN_SIZE_BASE = 0.1

# ???????? ??? ?????????
FEE_BPS = 8.0


In [ ]:
def fetch_spot_candles(symbol: str, candle_type: str = "1min", limit: int = 1200) -> pd.DataFrame:
    url = "https://api.kucoin.com/api/v1/market/candles"
    params = {"symbol": symbol, "type": candle_type}
    payload = requests.get(url, params=params, timeout=20).json()
    rows = payload["data"]

    df = pd.DataFrame(rows, columns=["ts", "open", "close", "high", "low", "volume", "turnover"])
    df["ts"] = df["ts"].astype("int64")
    for c in ["open", "close", "high", "low", "volume", "turnover"]:
        df[c] = df[c].astype("float64")

    df = df.sort_values("ts").reset_index(drop=True)
    return df.tail(limit).copy()


def fetch_futures_candles(symbol: str, granularity_min: int = 1, limit: int = 1200) -> pd.DataFrame:
    url = "https://api-futures.kucoin.com/api/v1/kline/query"
    params = {"symbol": symbol, "granularity": int(granularity_min)}
    payload = requests.get(url, params=params, timeout=20).json()
    rows = payload["data"]

    # ?????? futures: [time, open, high, low, close, volume, turnover]
    df = pd.DataFrame(rows, columns=["ts", "open", "high", "low", "close", "volume", "turnover"])
    df["ts"] = df["ts"].astype("int64")
    df["ts"] = np.where(df["ts"] > 10**12, df["ts"] // 1000, df["ts"])

    for c in ["open", "close", "high", "low", "volume", "turnover"]:
        df[c] = df[c].astype("float64")

    df = df.sort_values("ts").reset_index(drop=True)
    return df.tail(limit).copy()


spot = fetch_spot_candles(SPOT_SYMBOL, "1min", limit=LIMIT * 2)
fut = fetch_futures_candles(FUTURES_SYMBOL, TF_MIN, limit=LIMIT * 2)

df = pd.merge(
    spot[["ts", "open", "high", "low", "close", "volume"]].rename(
        columns={"open": "spot_open", "high": "spot_high", "low": "spot_low", "close": "spot_close", "volume": "spot_volume"}
    ),
    fut[["ts", "open", "high", "low", "close", "volume"]].rename(
        columns={"open": "fut_open", "high": "fut_high", "low": "fut_low", "close": "fut_close", "volume": "fut_volume"}
    ),
    on="ts",
    how="inner",
).sort_values("ts").reset_index(drop=True)

# ?? ?????? ?????? ????????? ????????? LIMIT ??????????
if len(df) > LIMIT:
    df = df.tail(LIMIT).reset_index(drop=True)

print("Merged rows:", len(df))
df.tail(3)


In [ ]:
# Basis / z-score ????????

df["basis"] = (df["fut_close"] - df["spot_close"]) / df["spot_close"]
df["basis_mean"] = df["basis"].rolling(BASIS_WINDOW).mean()
df["basis_std"] = df["basis"].rolling(BASIS_WINDOW).std(ddof=0)
df["basis_z"] = (df["basis"] - df["basis_mean"]) / (df["basis_std"] + 1e-8)

df["spread"] = df["fut_close"] - df["spot_close"]
df["spread_bps"] = 10000.0 * df["spread"] / df["spot_close"]

# ?????????? ?????????? ???? ??? ??????? step-PnL
# long spot: +ret, short futures: -ret

df["spot_ret_next"] = df["spot_close"].shift(-1) / df["spot_close"] - 1.0
df["fut_ret_next"] = df["fut_close"].shift(-1) / df["fut_close"] - 1.0

df = df.dropna().reset_index(drop=True)
print("Rows after features:", len(df))
df[["spot_close", "fut_close", "basis", "basis_z"]].tail(5)


In [ ]:
# ???????????? basis_z
plt.figure(figsize=(12, 4))
plt.plot(df["basis_z"].values, label="basis_z")
plt.axhline(ENTRY_Z, color="green", linestyle="--", label="entry_z")
plt.axhline(-ENTRY_Z, color="green", linestyle="--")
plt.axhline(EXIT_Z, color="orange", linestyle=":", label="exit_z")
plt.axhline(-EXIT_Z, color="orange", linestyle=":")
plt.title("Basis z-score")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
def build_pair_position(spot_price: float, max_spot_notional_usdt: float):
    raw_spot_qty = max_spot_notional_usdt / spot_price
    contracts = int(round(raw_spot_qty / CONTRACT_SIZE))
    contracts = max(contracts, 1)

    spot_qty = contracts * CONTRACT_SIZE
    if spot_qty < SPOT_MIN_SIZE_BASE:
        return 0.0, 0
    if spot_qty * spot_price < SPOT_MIN_FUNDS:
        return 0.0, 0

    return spot_qty, contracts


def run_rule_based_strategy(data: pd.DataFrame, entry_z: float, exit_z: float, max_spot_notional_usdt: float, fee_bps: float):
    fee_rate = fee_bps / 10000.0

    position = 0  # 0 = FLAT, 1 = LONG_SPOT_SHORT_FUT
    spot_qty = 0.0
    fut_contracts = 0
    equity = 0.0

    rows = []

    for i in range(len(data) - 1):
        row = data.iloc[i]
        nxt = data.iloc[i + 1]

        z = float(row["basis_z"])
        prev_position = position
        fee = 0.0

        # ????
        if position == 0 and z > entry_z:
            spot_qty_new, fut_contracts_new = build_pair_position(float(row["spot_close"]), max_spot_notional_usdt)
            if fut_contracts_new > 0:
                position = 1
                spot_qty = spot_qty_new
                fut_contracts = fut_contracts_new
                fee += fee_rate * (
                    spot_qty * row["spot_close"] + fut_contracts * CONTRACT_SIZE * row["fut_close"]
                )

        # ?????
        elif position == 1 and abs(z) < exit_z:
            fee += fee_rate * (
                spot_qty * row["spot_close"] + fut_contracts * CONTRACT_SIZE * row["fut_close"]
            )
            position = 0
            spot_qty = 0.0
            fut_contracts = 0

        # PnL ???????? ????
        pnl = 0.0
        if position == 1:
            fut_base_qty = -fut_contracts * CONTRACT_SIZE
            pnl += spot_qty * (nxt["spot_close"] - row["spot_close"])
            pnl += fut_base_qty * (nxt["fut_close"] - row["fut_close"])

        step_reward = pnl - fee
        equity += step_reward

        rows.append({
            "ts": int(row["ts"]),
            "z": z,
            "position": int(position),
            "position_change": int(position != prev_position),
            "step_reward": float(step_reward),
            "equity": float(equity),
        })

    return pd.DataFrame(rows)


In [ ]:
class DeltaNeutralEnv(gym.Env):
    def __init__(self, data: pd.DataFrame, max_spot_notional_usdt: float = 1.0, fee_bps: float = 8.0, lambda_delta: float = 0.1, lambda_turnover: float = 0.01):
        super().__init__()
        self.data = data.reset_index(drop=True)
        self.max_spot_notional_usdt = float(max_spot_notional_usdt)
        self.fee_rate = float(fee_bps) / 10000.0
        self.lambda_delta = float(lambda_delta)
        self.lambda_turnover = float(lambda_turnover)

        # 0=HOLD, 1=OPEN/KEEP, 2=CLOSE
        self.action_space = spaces.Discrete(3)

        # basis_z, basis, spread_bps, position, basis_std
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(5,), dtype=np.float32)

        self.reset()

    def _obs(self):
        row = self.data.iloc[self.i]
        return np.array([
            float(row["basis_z"]),
            float(row["basis"]),
            float(row["spread_bps"]),
            float(self.position),
            float(row["basis_std"]),
        ], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.i = 0
        self.position = 0
        self.spot_qty = 0.0
        self.fut_contracts = 0
        self.equity = 0.0
        return self._obs(), {}

    def step(self, action):
        row = self.data.iloc[self.i]
        nxt = self.data.iloc[self.i + 1]

        prev_position = self.position
        fee = 0.0
        turnover = 0.0

        # OPEN/KEEP
        if action == 1 and self.position == 0 and float(row["basis_z"]) > 0:
            spot_qty, fut_contracts = build_pair_position(float(row["spot_close"]), self.max_spot_notional_usdt)
            if fut_contracts > 0:
                self.position = 1
                self.spot_qty = spot_qty
                self.fut_contracts = fut_contracts
                turnover = 1.0
                fee += self.fee_rate * (
                    self.spot_qty * row["spot_close"] + self.fut_contracts * CONTRACT_SIZE * row["fut_close"]
                )

        # CLOSE
        if action == 2 and self.position == 1:
            fee += self.fee_rate * (
                self.spot_qty * row["spot_close"] + self.fut_contracts * CONTRACT_SIZE * row["fut_close"]
            )
            self.position = 0
            self.spot_qty = 0.0
            self.fut_contracts = 0
            turnover = 1.0

        pnl = 0.0
        fut_base_qty = 0.0
        if self.position == 1:
            fut_base_qty = -self.fut_contracts * CONTRACT_SIZE
            pnl += self.spot_qty * (nxt["spot_close"] - row["spot_close"])
            pnl += fut_base_qty * (nxt["fut_close"] - row["fut_close"])

        net_delta_notional = abs((self.spot_qty + fut_base_qty) * row["spot_close"])
        reward = pnl - fee - self.lambda_delta * net_delta_notional - self.lambda_turnover * turnover
        self.equity += reward

        self.i += 1
        terminated = self.i >= (len(self.data) - 1)
        truncated = False

        info = {
            "ts": int(row["ts"]),
            "position": int(self.position),
            "position_change": int(self.position != prev_position),
            "step_reward": float(reward),
            "equity": float(self.equity),
            "z": float(row["basis_z"]),
        }

        obs = self._obs() if not terminated else np.zeros((5,), dtype=np.float32)
        return obs, float(reward), terminated, truncated, info


In [ ]:
# Train/Test split
split_idx = int(len(df) * 0.7)
train_df = df.iloc[:split_idx].reset_index(drop=True)
test_df = df.iloc[split_idx:].reset_index(drop=True)

# Baseline ?? ?????
baseline_test = run_rule_based_strategy(
    test_df,
    entry_z=ENTRY_Z,
    exit_z=EXIT_Z,
    max_spot_notional_usdt=MAX_SPOT_NOTIONAL_USDT,
    fee_bps=FEE_BPS,
)

# RL ????????

def make_train_env():
    return DeltaNeutralEnv(train_df, max_spot_notional_usdt=MAX_SPOT_NOTIONAL_USDT, fee_bps=FEE_BPS)

vec_env = DummyVecEnv([make_train_env])
model = PPO(
    "MlpPolicy",
    vec_env,
    verbose=0,
    seed=SEED,
    learning_rate=3e-4,
    n_steps=256,
    batch_size=256,
    gamma=0.99,
)

model.learn(total_timesteps=25000)
print("RL training done")


In [ ]:
def run_rl(model, data: pd.DataFrame, max_spot_notional_usdt: float, fee_bps: float):
    env = DeltaNeutralEnv(data, max_spot_notional_usdt=max_spot_notional_usdt, fee_bps=fee_bps)
    obs, _ = env.reset()

    rows = []
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(int(action))
        done = bool(terminated or truncated)
        rows.append(info)

    return pd.DataFrame(rows)


def calc_metrics(bt: pd.DataFrame):
    if len(bt) == 0:
        return {"total_pnl": 0.0, "max_drawdown": 0.0, "sharpe_step": 0.0, "trades": 0}

    eq = bt["equity"].values
    rets = bt["step_reward"].values
    dd = eq - np.maximum.accumulate(eq)

    sharpe = float(np.mean(rets) / (np.std(rets) + 1e-9))

    return {
        "total_pnl": float(eq[-1]),
        "max_drawdown": float(dd.min()),
        "sharpe_step": sharpe,
        "trades": int(bt["position_change"].sum()),
    }


rl_test = run_rl(model, test_df, MAX_SPOT_NOTIONAL_USDT, FEE_BPS)

print("Baseline:", calc_metrics(baseline_test))
print("RL PPO  :", calc_metrics(rl_test))

plt.figure(figsize=(12, 5))
plt.plot(baseline_test["equity"].values, label="Baseline (basis z-score rules)")
plt.plot(rl_test["equity"].values, label="RL PPO")
plt.title("Equity Curve: Baseline vs RL")
plt.xlabel("Step")
plt.ylabel("Cumulative PnL (USDT)")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# ??????? JSON ??? ????????????? ????
# ??????: latest_forecast_signal_kucoin_rl.json

latest = df.iloc[-1]

state_json = {
    "signal_model": "basis_zscore",
    "timestamp": int(latest["ts"]),
    "spot_price": float(latest["spot_close"]),
    "futures_price": float(latest["fut_close"]),
    "basis": float(latest["basis"]),
    "basis_mean": float(latest["basis_mean"]),
    "basis_std": float(latest["basis_std"]),
    "basis_z": float(latest["basis_z"]),
    "entry_z": float(ENTRY_Z),
    "exit_z": float(EXIT_Z),
    "mode": "long_spot_short_futures_only",
}

out_path = "/content/latest_forecast_signal_kucoin_rl.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(state_json, f, ensure_ascii=False, indent=2)

print("Saved:", out_path)
print(json.dumps(state_json, ensure_ascii=False, indent=2))


In [ ]:
# ????????? JSON ?? Colab
from google.colab import files
files.download("/content/latest_forecast_signal_kucoin_rl.json")


## ??????? ??????? ? ????????? (Windows PowerShell)

```powershell
python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json "$HOME\Downloads\latest_forecast_signal_kucoin_rl.json"
```

???? ????? ??????? dry-run:

```powershell
python run_trade_signal.py --mode shadow --config config/micro_near_v1_1m.json --state-json "$HOME\Downloads\latest_forecast_signal_kucoin_rl.json"
```
